# Kaggle Corpus Preparation

Converts a parquet dataset into the pre-tokenized directory layout consumed by the
Nemotron finetuning notebook on Kaggle:

```
nemotron/
  tokens/{problem_id}/synthetic.json   ← {"tokens":[int,...], "mask":[0/1,...]}
  logprobs/index.jsonl                 ← {"epoch":0, "step":i, "problem_id":...}
```

**Two modes** — select with `DATASET_MODE` in Cell 1:

| Mode | Dataset | `answer` col | Assistant tail |
|------|---------|-------------|----------------|
| `"prediction"` | `result_examples_0_result_k_6_balanced_cot.parquet` | required | `</think>\n\boxed{answer}\n<\|im_end\|>` |
| `"analysis"` | `result_examples_0_analysis_cot_unbalanced.parquet` | absent | `</think>\n<\|im_end\|>` (no boxed) |

**Pipeline:**
1. Shuffle all rows with a fixed seed.
2. Tokenize each row via the Nemotron chat template:
   - User turn = original `prompt` (+ boxed-answer instruction in prediction mode).
   - Loss mask = 0 over:
     - the user turn (up to and including the open `<think>` tag), **and**
     - the fixed boilerplate prefix inside the CoT (22-op list + 3-strategy description,
       byte-identical across all rows, ~63 % of average CoT ≈ 800 tokens).
   - Loss mask = 1 from the first row-specific line through the full assistant turn.
     - prediction: `{reasoning}</think>\n\boxed{answer}\n<|im_end|>`
     - analysis: `{reasoning}</think>\n<|im_end|>`
   - Tokens remain in the sequence in all cases — the model conditions on the boilerplate;
     only the gradient is suppressed over it.
3. Write one `synthetic.json` per row (unique `problem_id = {id}__{i:06d}`).
4. Write `index.jsonl` in shuffle order — this becomes the training order.
5. (Optional) upload as a Kaggle dataset.

**Training notebook pointers** (update these constants before training):
```python
CORPUS_PATH      = "/kaggle/input/datasets/<user>/nemotron-custom-corpus/nemotron/tokens"
TRAIN_ORDER_PATH = "/kaggle/input/datasets/<user>/nemotron-custom-corpus/nemotron/logprobs/index.jsonl"
```

## Cell 1 — Configuration

In [ ]:
import os

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

# ── Dataset mode ──────────────────────────────────────────────────────────────
# "prediction" : balanced_cot dataset with an `answer` column; emits \boxed{}.
# "analysis"   : analysis_cot dataset with NO `answer` column; no \boxed{}.
DATASET_MODE = "analysis"   # ← change this to switch datasets

_MODE_CFG = {
    "prediction": {
        "in_local":  "data/result_examples_0_result_k_6_balanced_cot.parquet",
        "in_kaggle": "/kaggle/input/datasets/zosov07/result-examples-0-result-k-6-balanced-cot/result_examples_0_result_k_6_balanced_cot.parquet",
        "marker":     "Formatted equation:",
        "emit_boxed": True,
        "required":   {"id", "prompt", "cot", "answer"},
    },
    "analysis": {
        "in_local":  "data/result_examples_0_analysis_cot_unbalanced.parquet",
        # Fill in the Kaggle dataset slug after uploading the analysis parquet.
        "in_kaggle": "/kaggle/input/datasets/zosov07/<analysis-dataset-slug>/result_examples_0_analysis_cot_unbalanced.parquet",
        "marker":     "Shown equation:",
        "emit_boxed": False,
        "required":   {"id", "prompt", "cot"},
    },
}

assert DATASET_MODE in _MODE_CFG, f"Unknown DATASET_MODE {repr(DATASET_MODE)}; choose 'prediction' or 'analysis'"
_cfg             = _MODE_CFG[DATASET_MODE]
IN_PARQUET       = _cfg["in_kaggle"] if IS_KAGGLE else _cfg["in_local"]
COT_CONST_MARKER = _cfg["marker"]        # first row-specific line → mask=1 starts here
EMIT_BOXED       = _cfg["emit_boxed"]    # whether to append \boxed{answer} after </think>
REQUIRED_COLS    = _cfg["required"]

# ── Tokenizer source ──────────────────────────────────────────────────────────
# On Kaggle: downloaded via kagglehub (same source as the training notebook).
# Locally: set to a local path or the HF Hub string.
TOKENIZER_KAGGLE_MODEL = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
TOKENIZER_HF_FALLBACK  = "unsloth/Nemotron-3-Nano-30B-A3B"

# ── Boxed-answer instruction appended to every user turn (prediction mode only) ──
# Must match what the competition metric harness sends at inference time.
BOXED_INSTRUCTION = (
    "Please put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

# ── Output ────────────────────────────────────────────────────────────────────
# /kaggle/working/ is the notebook's output directory; its contents are saved
# as a dataset when the notebook is committed with Save Version.
OUT_ROOT   = "/kaggle/working/nemotron" if IS_KAGGLE else "/tmp/nemotron_corpus_test"
TOKENS_DIR = os.path.join(OUT_ROOT, "tokens")
INDEX_PATH = os.path.join(OUT_ROOT, "logprobs", "index.jsonl")

# ── Shuffle seed ──────────────────────────────────────────────────────────────
SEED = 0

# ── Kaggle dataset to upload to (used in optional upload cell) ────────────────
# Format: "<username>/<dataset-name>"  (slug, lower-case-with-hyphens)
KAGGLE_DATASET = "<user>/nemotron-custom-corpus"

# ── CoT loss masking ─────────────────────────────────────────────────────────
# The CoT always opens with a fixed boilerplate block (22-op list + 3-strategy
# description) that is byte-identical across all rows.  Setting True keeps the
# tokens in the sequence but zeroes the loss over them. mask=1 resumes at the
# first row-specific line identified by COT_CONST_MARKER.
MASK_COT_CONST_BLOCK = True

print(f"DATASET_MODE        : {DATASET_MODE}")
print(f"IS_KAGGLE           : {IS_KAGGLE}")
print(f"IN_PARQUET          : {IN_PARQUET}")
print(f"COT_CONST_MARKER    : {repr(COT_CONST_MARKER)}")
print(f"EMIT_BOXED          : {EMIT_BOXED}")
print(f"OUT_ROOT            : {OUT_ROOT}")
print(f"SEED                : {SEED}")
print(f"MASK_COT_CONST_BLOCK: {MASK_COT_CONST_BLOCK}")

## Cell 2 — Install missing packages (Kaggle guard)

In [ ]:
import importlib, subprocess, sys

def _ensure(package, import_name=None):
    """pip-install `package` if `import_name` (default=package) cannot be imported."""
    import_name = import_name or package
    try:
        importlib.import_module(import_name)
    except ModuleNotFoundError:
        print(f"Installing {package} ...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

_ensure("transformers")
_ensure("tqdm")
# pandas and kagglehub are pre-installed on Kaggle; install locally if missing
_ensure("pandas")
_ensure("kagglehub")

print("All packages available.")

## Cell 3 — Load tokenizer

In [ ]:
from transformers import AutoTokenizer

if IS_KAGGLE:
    import kagglehub
    tokenizer_path = kagglehub.model_download(TOKENIZER_KAGGLE_MODEL)
    print(f"Tokenizer downloaded to: {tokenizer_path}")
else:
    tokenizer_path = TOKENIZER_HF_FALLBACK
    print(f"Using tokenizer: {tokenizer_path}")

tok = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True)
print(f"Vocab size      : {tok.vocab_size:,}")
print(f"Chat template   : {'yes' if tok.chat_template else 'NO — will fail'}")

# The Nemotron template renders the assistant turn as:
#   <|im_start|>assistant\n<think></think>{content}<|im_end|>
#
# We rearrange to one of two forms depending on EMIT_BOXED:
#   prediction: <|im_start|>assistant\n<think>\n{cot} \n</think>\n\boxed{answer}\n<|im_end|>
#   analysis:   <|im_start|>assistant\n<think>\n{cot} \n</think>\n<|im_end|>
#
# The loss-mask prefix boundary sits right after the opening <think> tag so the
# model is trained to generate everything from the leading \n onwards.

def _rearrange_think(full_text: str, cot: str, answer: str | None) -> tuple[str, str]:
    """Return (full_text_fixed, prefix_text).

    full_text_fixed:
        Assistant turn with {cot} inside <think>...</think>.
        If EMIT_BOXED: \\boxed{answer} follows on its own line before <|im_end|>.
        If not EMIT_BOXED: </think> is followed directly by <|im_end|> (no boxed).
    prefix_text:
        Ends at (and includes) the opening <think> tag — this is where mask=0
        ends and mask=1 begins.
    """
    EMPTY_THINK = "<think></think>"
    assert EMPTY_THINK in full_text, \
        f"Expected '<think></think>' in rendered text; got: {repr(full_text[-80:])}"
    idx    = full_text.rindex(EMPTY_THINK)
    before = full_text[:idx]                               # up to <think></think>
    after  = full_text[idx + len(EMPTY_THINK):]           # anything after </think>
    im_end = after.index("<|im_end|>")
    tail   = after[im_end:]                               # <|im_end|>\n

    if EMIT_BOXED:
        assert answer is not None, "answer must be provided when EMIT_BOXED=True"
        full_fixed = (before + "<think>\n" + cot
                      + " \n</think>\n\\boxed{" + answer + "}\n" + tail)
    else:
        full_fixed = before + "<think>\n" + cot + " \n</think>\n" + tail

    prefix_text = before + "<think>"                       # open <think>, not closed
    return full_fixed, prefix_text

# Sanity check with toy messages
_full_msg = [{"role": "user",      "content": "hello"},
             {"role": "assistant", "content": "world"}]
_full_raw   = tok.apply_chat_template(_full_msg, tokenize=False, enable_thinking=False)
_full_fixed, _prefix_text = _rearrange_think(_full_raw, "world", "42" if EMIT_BOXED else None)

# Structural assertions
assert "<think>\nworld \n</think>" in _full_fixed, \
    f"Expected leading/trailing newlines around cot inside <think>; got: {repr(_full_fixed)}"
if EMIT_BOXED:
    assert "\\boxed{42}" in _full_fixed, \
        f"Expected \\boxed{{42}} in full_fixed; got: {repr(_full_fixed)}"
    _im_end_pos = _full_fixed.rindex("<|im_end|>")
    _box_pos    = _full_fixed.rindex("\\boxed{42}")
    assert _box_pos < _im_end_pos, "\\boxed{answer} must appear before <|im_end|>"
else:
    assert "\\boxed{" not in _full_fixed, \
        f"\\boxed{{ should NOT appear in analysis mode; got: {repr(_full_fixed[-80:])}"
    assert "</think>\n<|im_end|>" in _full_fixed, \
        f"Expected '</think>\\n<|im_end|>' in analysis mode; got: {repr(_full_fixed[-80:])}"

# Token-prefix assertion
_prefix = tok.encode(_prefix_text)
_full   = tok.encode(_full_fixed)
assert isinstance(_prefix, list) and isinstance(_prefix[0], int)
assert _full[:len(_prefix)] == _prefix, \
    "Prefix is not a clean prefix of the full sequence after rearrangement"

print("apply_chat_template boundary check: OK")
print(f"Mode        : {DATASET_MODE}  (EMIT_BOXED={EMIT_BOXED})")
print(f"Prefix end  : {repr(_prefix_text[-60:])}")
print(f"Full end    : {repr(_full_fixed[-80:])}")

## Cell 4 — Load parquet and shuffle

In [ ]:
import pandas as pd

df = pd.read_parquet(IN_PARQUET)
print(f"Loaded {len(df):,} rows  |  columns: {list(df.columns)}")

# Guard: ensure required columns are present
_missing = REQUIRED_COLS - set(df.columns)
assert not _missing, f"Missing required columns: {_missing}"
assert df["cot"].isna().sum() == 0, "cot column contains nulls"
if "answer" in df.columns:
    assert df["answer"].isna().sum() == 0, "answer column contains nulls"
print(f"Required columns present and non-null. Unique ids: {df['problem_id'].nunique():,}")

df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print(f"Shuffled (seed={SEED}). First 3 ids: {list(df['problem_id'].head(3))}")

## Cell 5 — Tokenise helper

In [ ]:
def render_example(prompt: str, cot: str, answer: str | None) -> tuple[list[int], list[int]]:
    """Return (tokens, mask) for one training example.

    prediction mode (EMIT_BOXED=True):
        User turn = prompt + BOXED_INSTRUCTION.
        Assistant turn = <think>{cot}</think>\\n\\boxed{answer}\\n<|im_end|>
    analysis mode (EMIT_BOXED=False):
        User turn = prompt (no boxed instruction).
        Assistant turn = <think>{cot}</think>\\n<|im_end|>  (no boxed answer)

    Loss mask (mask=0 → no gradient, mask=1 → backpropagate):
      - User turn + assistant header up to and including the opening <think> tag:
        always mask=0.
      - If MASK_COT_CONST_BLOCK is True: the fixed boilerplate (22-op list +
        3-strategy description) appearing before COT_CONST_MARKER is mask=0.
        Tokens remain in the sequence; only the loss is suppressed.
      - From COT_CONST_MARKER onward through </think>[\\n\\boxed{answer}\\n]<|im_end|>:
        mask=1.
    """
    user_content = f"{prompt}\n{BOXED_INSTRUCTION}" if EMIT_BOXED else prompt
    full_turn = [{"role": "user",      "content": user_content},
                 {"role": "assistant", "content": cot}]

    full_raw = tok.apply_chat_template(full_turn, tokenize=False, enable_thinking=False)
    full_fixed, prefix_text = _rearrange_think(full_raw, cot, answer)

    prefix_ids: list[int] = tok.encode(prefix_text)
    full_ids:   list[int] = tok.encode(full_fixed)

    assert full_ids[:len(prefix_ids)] == prefix_ids, (
        "Prefix mismatch after rearrangement.\n"
        f"prefix end : {repr(prefix_text[-40:])}\n"
        f"full start : {repr(full_fixed[:len(prefix_text)+20])}"
    )

    n_prefix = len(prefix_ids)   # end of user turn + opening <think> tag

    if MASK_COT_CONST_BLOCK:
        # Locate the first row-specific line in the rendered text.
        const_end = full_fixed.find(COT_CONST_MARKER)
        if const_end != -1:
            # Tokenise everything up to that marker.  The boundary may fall
            # mid-token (the last constant token could partially overlap the
            # marker), so we walk the common prefix rather than asserting
            # exact equality — one ambiguous boundary token is kept in the
            # trainable region, which is harmless.
            const_prefix_ids = tok.encode(full_fixed[:const_end])
            n_mask = 0
            for a, b in zip(full_ids, const_prefix_ids):
                if a != b:
                    break
                n_mask += 1
            n_mask = max(n_mask, n_prefix)   # never mask less than the user turn
        else:
            # Marker not found (non-template CoT): fall back to the original boundary.
            n_mask = n_prefix
    else:
        n_mask = n_prefix

    mask = [0] * n_mask + [1] * (len(full_ids) - n_mask)
    assert len(full_ids) == len(mask)
    assert any(mask), "all mask tokens are 0 — no trainable signal"
    return full_ids, mask

# Quick smoke test on row 0
_r      = df.iloc[0]
_answer = _r.get("answer") if "answer" in df.columns else None
_toks, _mask = render_example(_r["prompt"], _r["cot"], _answer)
n_masked = sum(1 for m in _mask if m == 0)
n_train  = sum(1 for m in _mask if m == 1)
print(f"Row 0 — total tokens: {len(_toks)}, masked (no loss): {n_masked}, trainable: {n_train}")
masked_decoded = tok.decode(_toks[:n_masked], skip_special_tokens=False)
train_decoded  = tok.decode(_toks[n_masked:], skip_special_tokens=False)
print("Masked end    (last 60) :", repr(masked_decoded[-60:]))
print("Trainable start (first 40):", repr(train_decoded[:40]))
print("Trainable end   (last 60):", repr(train_decoded[-60:]))

# Assertions
assert "Let's assume the following set of available operations" in masked_decoded, \
    "OPS_BLOCK opener not found in the masked region"
if MASK_COT_CONST_BLOCK:
    assert COT_CONST_MARKER not in masked_decoded, \
        f"{repr(COT_CONST_MARKER)} should NOT appear in the masked region"
    assert train_decoded.startswith(COT_CONST_MARKER), \
        f"Trainable region should start with {repr(COT_CONST_MARKER)}; got {repr(train_decoded[:40])}"
if EMIT_BOXED:
    assert "\\boxed{" in train_decoded, "\\boxed{ not found in trainable region"
    assert _answer in train_decoded,    "answer string not found in trainable region"
else:
    assert "\\boxed{" not in train_decoded, \
        f"\\boxed{{ must NOT appear in analysis mode; found in: {repr(train_decoded[-60:])}"
# Confirm a single 0→1 transition (clean boundary)
_transitions = sum(1 for i in range(1, len(_mask)) if _mask[i] != _mask[i-1])
assert _transitions == 1, f"Expected 1 mask transition; got {_transitions}"
print("Smoke test: boundary assertions OK")

## Cell 6 — Emit corpus (tokens + index)

In [ ]:
import json
from tqdm.auto import tqdm

os.makedirs(TOKENS_DIR, exist_ok=True)
os.makedirs(os.path.dirname(INDEX_PATH), exist_ok=True)

index_records: list[dict] = []

for i, row in enumerate(tqdm(df.itertuples(index=False), total=len(df),
                              desc="Tokenising")):
    # Unique problem_id: original id + zero-padded shuffle position.
    # The training notebook dedups by problem_id so uniqueness is required.
    # The original id is kept as the prefix so puzzle-level splits can
    # use .split("__")[0] to recover it.
    pid = f"{row.id}__{i:06d}"

    # answer is None in analysis mode (no answer column); getattr is safe for both.
    answer = getattr(row, "answer", None)
    tokens, mask = render_example(row.prompt, row.cot, answer)

    # Write tokens/{pid}/synthetic.json
    seg_dir = os.path.join(TOKENS_DIR, pid)
    os.makedirs(seg_dir, exist_ok=True)
    with open(os.path.join(seg_dir, "synthetic.json"), "w") as f:
        json.dump({"problem_id": pid, "original_id": row.id, "tokens": tokens, "mask": mask}, f)

    index_records.append({"epoch": 0, "step": i, "problem_id": pid})

# Write logprobs/index.jsonl
with open(INDEX_PATH, "w") as f:
    for rec in index_records:
        f.write(json.dumps(rec) + "\n")

print(f"\nWrote {len(index_records):,} examples to {OUT_ROOT}")
print(f"  tokens/   : {TOKENS_DIR}")
print(f"  index     : {INDEX_PATH}")

## Cell 7 — Validation

In [ ]:
import random

# 1. Directory count
written_dirs = [d for d in os.listdir(TOKENS_DIR)
                if os.path.isdir(os.path.join(TOKENS_DIR, d))]
assert len(written_dirs) == len(df), (
    f"Directory count mismatch: {len(written_dirs)} dirs vs {len(df)} rows"
)
print(f"[OK] {len(written_dirs):,} token directories written")

# 2. index.jsonl line count and all problem_ids present
with open(INDEX_PATH) as f:
    idx_lines = [json.loads(l) for l in f]
assert len(idx_lines) == len(df), (
    f"index.jsonl has {len(idx_lines)} lines but {len(df)} examples expected"
)
for rec in idx_lines:
    seg_path = os.path.join(TOKENS_DIR, rec["problem_id"], "synthetic.json")
    assert os.path.isfile(seg_path), f"Missing synthetic.json for {rec['problem_id']}"
print(f"[OK] index.jsonl: {len(idx_lines):,} lines, all problem_ids resolve to files")

# 3. Random sample: tokens/mask integrity + masking boundary
rng = random.Random(99)
sample_pids = rng.sample([r["problem_id"] for r in idx_lines], k=min(200, len(idx_lines)))
total_tokens    = 0
total_unmasked  = 0
max_seq_len     = 0
over_8192       = 0

for pid in sample_pids:
    seg = json.load(open(os.path.join(TOKENS_DIR, pid, "synthetic.json")))
    toks, msk = seg["tokens"], seg["mask"]
    assert len(toks) == len(msk), f"{pid}: len mismatch"
    assert any(msk),              f"{pid}: all mask=0"
    # Single 0→1 transition required (clean boundary)
    transitions = sum(1 for i in range(1, len(msk)) if msk[i] != msk[i-1])
    assert transitions == 1, f"{pid}: expected 1 mask transition, got {transitions}"
    # Locate trainable boundary
    n_train_start = msk.index(1)
    masked_text = tok.decode(toks[:n_train_start], skip_special_tokens=False)
    train_text  = tok.decode(toks[n_train_start:], skip_special_tokens=False)
    # Ops-block opener must be in the masked region
    assert "Let's assume the following set of available operations" in masked_text, \
        f"{pid}: OPS_BLOCK opener not found in masked region"
    # If masking is active, marker must not appear in the masked region
    if MASK_COT_CONST_BLOCK:
        assert COT_CONST_MARKER not in masked_text, \
            f"{pid}: {repr(COT_CONST_MARKER)} found in masked region — boundary too late"
        assert train_text.startswith(COT_CONST_MARKER), \
            f"{pid}: trainable region does not start with {repr(COT_CONST_MARKER)}"
    # Boxed answer check: required in prediction mode, forbidden in analysis mode
    if EMIT_BOXED:
        assert "\\boxed{" in train_text, (
            f"{pid}: \\boxed{{ not found in trainable region — "
            f"train start: {repr(train_text[:80])}"
        )
    else:
        assert "\\boxed{" not in train_text, (
            f"{pid}: \\boxed{{ must NOT appear in analysis mode — "
            f"train end: {repr(train_text[-80:])}"
        )

# Full pass for stats
for rec in tqdm(idx_lines, desc="Stats pass"):
    seg = json.load(open(os.path.join(TOKENS_DIR, rec["problem_id"], "synthetic.json")))
    n = len(seg["tokens"])
    total_tokens   += n
    total_unmasked += sum(seg["mask"])
    if n > max_seq_len:
        max_seq_len = n
    if n > 8192:
        over_8192 += 1

print(f"[OK] 200-sample integrity + masking-boundary check passed")
print(f"\nCorpus stats ({DATASET_MODE} mode):")
print(f"  Total tokens     : {total_tokens:,}")
print(f"  Unmasked (train) : {total_unmasked:,.0f}  ({100*total_unmasked/total_tokens:.1f}%)")
print(f"  Max seq length   : {max_seq_len}")
if MASK_COT_CONST_BLOCK:
    masked_pct = 100 * (1 - total_unmasked / total_tokens)
    print(f"  CoT const block  : masked (OPS+STRATEGIES boilerplate, ~{masked_pct:.0f}% suppressed)")
if over_8192:
    print(f"  ⚠️  {over_8192} sequences > 8192 tokens — training notebook will truncate them")
else:
    print(f"  All sequences ≤ 8192 tokens ✓")

# 4. Decode a random example to eyeball the mask boundary
spot_pid = rng.choice(sample_pids)
spot = json.load(open(os.path.join(TOKENS_DIR, spot_pid, "synthetic.json")))
n_train_spot = spot["mask"].index(1)        # first 1 = start of trainable region
print(f"\nSpotcheck pid   : {spot_pid}")
print("Masked end      :", repr(tok.decode(spot["tokens"][:n_train_spot], skip_special_tokens=False)[-120:]))
print("Trainable start :", repr(tok.decode(spot["tokens"][n_train_spot:], skip_special_tokens=False)[:120]))
print("Trainable end   :", repr(tok.decode(spot["tokens"][n_train_spot:], skip_special_tokens=False)[-80:]))

## Cell 8 — Training-notebook round-trip check

Verifies that the loader logic in the training notebook produces valid
`input / target / weight` triples from our files.

In [ ]:
# Simulate the training notebook's corpus loader (IS_KAGGLE branch) for 5 examples.
_MAX_SEQ_LEN = 8192  # matches the training notebook default

examples_check = []
for rec in idx_lines[:5]:
    seg_path = os.path.join(TOKENS_DIR, rec["problem_id"], "synthetic.json")
    seg = json.load(open(seg_path))
    tokens = seg["tokens"]
    mask   = seg["mask"]
    # Exact logic from the training notebook
    if len(tokens) > _MAX_SEQ_LEN:
        tokens = tokens[:_MAX_SEQ_LEN]
        mask   = mask[:_MAX_SEQ_LEN]
    if not any(mask):
        print(f"SKIP (all masked): {rec['problem_id']}")
        continue
    examples_check.append({
        "problem_id": rec["problem_id"],
        "tokens":  tokens[:-1],
        "targets": tokens[1:],
        "weights": [float(m) for m in mask[1:]],
    })

for ex in examples_check:
    assert len(ex["tokens"]) == len(ex["targets"]) == len(ex["weights"]), \
        f"{ex['problem_id']}: length mismatch in training triple"
    assert sum(ex["weights"]) > 0, f"{ex['problem_id']}: no unmasked tokens after slice"

print(f"[OK] Round-trip check passed for {len(examples_check)} examples")
for ex in examples_check:
    print(f"  {ex['problem_id']}: seq_len={len(ex['tokens'])}, "
          f"unmasked={sum(ex['weights']):.0f}")

## Cell 9 — Save as Kaggle dataset

Two routes:

**Route A (API upload — runs from inside the notebook):** writes
`dataset-metadata.json` and calls `kaggle datasets version`.  
Requires the `KAGGLE_API_TOKEN` env var (or `~/.kaggle/kaggle.json`).

**Route B (Save Version):** leave files under `/kaggle/working/nemotron/` and
commit the notebook as a *Save Version*. The output is attached automatically
and can be turned into a dataset from the notebook output page. No extra code
needed — just commit the notebook and add the output as a dataset.

In [ ]:
# ── Route A: Kaggle API upload ─────────────────────────────────────────────────
# Set UPLOAD_VIA_API = True to activate. Requires kaggle.json or KAGGLE_API_TOKEN.
UPLOAD_VIA_API = False

if UPLOAD_VIA_API:
    import subprocess, tempfile, shutil

    # Write Kaggle credentials if supplied via env var
    if "KAGGLE_API_TOKEN" in os.environ:
        kaggle_dir = os.path.expanduser("~/.kaggle")
        os.makedirs(kaggle_dir, exist_ok=True)
        with open(os.path.join(kaggle_dir, "kaggle.json"), "w") as _f:
            _f.write(os.environ["KAGGLE_API_TOKEN"])
        os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)

    # Create a staging directory with metadata so kaggle CLI can upload it.
    upload_dir = tempfile.mkdtemp(prefix="kaggle_upload_")
    # Copy the whole nemotron/ tree into the staging dir.
    shutil.copytree(OUT_ROOT, os.path.join(upload_dir, "nemotron"))

    dataset_meta = {
        "id": KAGGLE_DATASET,
        "title": KAGGLE_DATASET.split("/")[1],
        "licenses": [{"name": "other"}],
    }
    with open(os.path.join(upload_dir, "dataset-metadata.json"), "w") as _f:
        json.dump(dataset_meta, _f)

    print(f"Uploading {upload_dir} → {KAGGLE_DATASET} ...")
    # First-time: use `kaggle datasets create`; subsequent runs: `version`.
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", upload_dir,
         "-m", f"corpus seed={SEED}, {len(df)} examples"],
        capture_output=True, text=True
    )
    if result.returncode != 0 and "404" in result.stderr:
        print("Dataset not found — creating new dataset...")
        subprocess.run(
            ["kaggle", "datasets", "create", "-p", upload_dir],
            check=True
        )
    else:
        print(result.stdout or result.stderr)
    shutil.rmtree(upload_dir)
    print("Upload complete.")
else:
    print("UPLOAD_VIA_API=False — skipping API upload.")
    print()
    print("Route B (Save Version): commit this notebook.")
    print("The output at /kaggle/working/nemotron/ will be saved automatically.")
    print("From the output page, click 'New Dataset' to publish it.")
    print()
    print("After publishing, update the training notebook constants:")
    owner = KAGGLE_DATASET.split('/')[0]
    slug  = KAGGLE_DATASET.split('/')[1]
    print(f'  CORPUS_PATH      = "/kaggle/input/datasets/{owner}/{slug}/nemotron/tokens"')
    print(f'  TRAIN_ORDER_PATH = "/kaggle/input/datasets/{owner}/{slug}/nemotron/logprobs/index.jsonl"')